# NYC Taxi Incremental ETL Pipeline

Incremental PySpark ETL for NYC Yellow Taxi trip records.  
Processes only new files on each run, enriches with zone names, and writes a deduplicated cumulative output dataset.

**Custom Scenario:** `passenger_count` values that are null or 0 are imputed with the median
passenger count for the same calendar month, computed from valid rows in the same file.

## 1. Imports & Setup

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from functools import reduce
from pathlib import Path

# Set working directory to project root inside the container
os.chdir("/home/jovyan/work")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

_pipeline_start = time.time()

## 2. Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("NYC Taxi Incremental ETL")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("Spark UI:      http://localhost:4040")

## 3. Configuration

In [ ]:
INBOX_DIR        = "data/inbox"
OUTBOX_DIR       = "data/outbox"
OUTBOX_PATH      = "data/outbox/trips_enriched.parquet"
MANIFEST_PATH    = "state/manifest.json"
ZONE_LOOKUP_PATH = "data/inbox/taxi_zone_lookup.parquet"

# Dedup key -- _fare_rounded avoids float precision issues in equality comparison
DEDUP_KEY = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "_fare_rounded",
]

## 4. Incremental State (Manifest)

The manifest (`state/manifest.json`) records every processed input file.
On each run the job reads the manifest, skips already-processed filenames, and appends
new entries only after the output has been written. This makes reruns fully idempotent.

In [ ]:
def load_manifest(path):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return {"processed_files": []}

def save_manifest(path, manifest):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(manifest, f, indent=2)

manifest = load_manifest(MANIFEST_PATH)
processed_filenames = {entry["filename"] for entry in manifest["processed_files"]}
print(f"Already processed: {processed_filenames or 'none'}")

## 5. Scan Inbox for New Files

In [ ]:
all_trip_files = sorted([
    f for f in Path(INBOX_DIR).glob("*.parquet")
    if f.name != "taxi_zone_lookup.parquet"
])

new_files = [f for f in all_trip_files if f.name not in processed_filenames]

print(f"Trip files in inbox : {len(all_trip_files)}")
print(f"Already processed   : {len(processed_filenames)}")
print(f"New files to process: {len(new_files)}")
for f in new_files:
    print(f"  - {f.name}")

if not new_files:
    print("\nNo new files to process. Job is up to date.")

## 6. Load Zone Lookup Table

265 rows, ~7 KB.

In [ ]:
zone_lookup = spark.read.parquet(ZONE_LOOKUP_PATH)
zone_lookup.cache()
print(f"Zone lookup rows: {zone_lookup.count()}")
zone_lookup.show(5, truncate=False)

## 7. Process New Files

### Cleaning Rules

| # | Field | Rule | Action |
|---|-------|------|--------|
| 1 | `trip_distance` | null or <= 0 | Drop row |
| 2 | `fare_amount` | null or < 0 | Drop row |
| 3 | `PULocationID` / `DOLocationID` | null | Drop row |
| 4 | `tpep_pickup_datetime` / `tpep_dropoff_datetime` | null | Drop row |
| 5 | `tpep_dropoff_datetime` | <= `tpep_pickup_datetime` | Drop row |
| 6 | `passenger_count` | null or 0 | **Impute** with per-month median (Custom Scenario) |

### Deduplication Key

`(tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID, fare_amount)`

`fare_amount` is rounded to 2 decimal places before comparison to avoid floating-point equality issues.

In [ ]:
ingested_at = datetime.now(timezone.utc).isoformat()
new_manifest_entries = []
processed_dfs = []
final_count = 0
SEP = "=" * 60

for file_path in new_files:
    filename  = file_path.name
    file_size = file_path.stat().st_size
    print(f"\n{SEP}")
    print(f"Processing: {filename}  ({file_size / 1e6:.1f} MB)")
    print(SEP)

    # --- Read ---
    df = spark.read.parquet(str(file_path))
    row_count_raw = df.count()
    print(f"  Raw rows: {row_count_raw:,}")

    # --- Cast types ---
    df = (df
        .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("passenger_count",       F.col("passenger_count").cast("integer"))
        .withColumn("trip_distance",         F.col("trip_distance").cast("double"))
        .withColumn("fare_amount",           F.col("fare_amount").cast("double"))
        .withColumn("PULocationID",          F.col("PULocationID").cast("integer"))
        .withColumn("DOLocationID",          F.col("DOLocationID").cast("integer"))
    )

    # --- Cleaning: all rules combined in one filter pass ---
    df = df.filter(
        F.col("trip_distance").isNotNull()        & (F.col("trip_distance") > 0)    &
        F.col("fare_amount").isNotNull()          & (F.col("fare_amount") >= 0)     &
        F.col("PULocationID").isNotNull()         & F.col("DOLocationID").isNotNull() &
        F.col("tpep_pickup_datetime").isNotNull() &
        F.col("tpep_dropoff_datetime").isNotNull() &
        (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    )
    df.cache()
    row_count_after_clean = df.count()
    print(f"  After cleaning: {row_count_after_clean:,}  (removed {row_count_raw - row_count_after_clean:,})")

    # --- Passenger count imputation (Custom Scenario) ---
    # Medians computed only from valid rows (passenger_count > 0, not null) in this file
    valid_pc = df.filter(F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0))

    month_medians = (
        valid_pc
        .withColumn("_month", F.month("tpep_pickup_datetime"))
        .groupBy("_month")
        .agg(F.percentile_approx("passenger_count", 0.5).alias("_month_median"))
    )

    # Global median fallback: used if a calendar month has no valid rows at all
    gm_row = valid_pc.agg(F.percentile_approx("passenger_count", 0.5).alias("gm")).collect()
    global_median = gm_row[0]["gm"] if gm_row and gm_row[0]["gm"] is not None else 1

    impute_mask = F.col("passenger_count").isNull() | (F.col("passenger_count") == 0)
    imputed_count = df.filter(impute_mask).count()

    print(f"\n  Passenger count imputation:")
    for row in month_medians.orderBy("_month").collect():
        print(f"    Month {row['_month']:>2}: median = {row['_month_median']}")
    print(f"    Global fallback median : {global_median}")
    print(f"    Rows imputed           : {imputed_count:,}")

    # Apply imputation: join month medians back, fill null/0 values
    df = df.withColumn("_month", F.month("tpep_pickup_datetime"))
    df = df.join(month_medians, on="_month", how="left")
    df = df.withColumn(
        "passenger_count",
        F.when(impute_mask, F.coalesce(F.col("_month_median"), F.lit(global_median)))
         .otherwise(F.col("passenger_count"))
    )
    df = df.drop("_month", "_month_median")

    # --- Deduplication ---
    df = df.withColumn("_fare_rounded", F.round(F.col("fare_amount"), 2))
    df = df.dropDuplicates(DEDUP_KEY)
    df = df.drop("_fare_rounded")
    row_count_after_dedup = df.count()
    print(f"\n  After dedup: {row_count_after_dedup:,}  (removed {row_count_after_clean - row_count_after_dedup:,})")

    # --- Derived fields & metadata ---
    df = (df
        .withColumn(
            "trip_duration_minutes",
            F.round(
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
                2
            )
        )
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
        .withColumn("source_file", F.lit(filename))
        .withColumn("ingested_at", F.lit(ingested_at))
    )

    # Release the cached cleaned DataFrame — no longer needed after dedup/enrichment
    df.unpersist()

    processed_dfs.append(df)
    new_manifest_entries.append({
        "filename":                filename,
        "file_size_bytes":         file_size,
        "row_count_raw":           row_count_raw,
        "row_count_after_clean":   row_count_after_clean,
        "row_count_after_dedup":   row_count_after_dedup,
        "imputed_passenger_count": imputed_count,
        "global_median_used":      int(global_median),
        "processed_at":            ingested_at,
    })

if new_files:
    print(f"\n{SEP}")
    print("All new files processed.")
    print(SEP)
else:
    print("No new files -- add files to data/inbox/ and rerun.")

## 8. Bad Row Examples

Re-reads the first newly processed raw file to surface examples of each cleaning rule.
Copy the output into the README.

In [ ]:
if new_files:
    raw_df = spark.read.parquet(str(new_files[0]))
    raw_df = (raw_df
        .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("passenger_count",       F.col("passenger_count").cast("integer"))
        .withColumn("trip_distance",         F.col("trip_distance").cast("double"))
        .withColumn("fare_amount",           F.col("fare_amount").cast("double"))
    )
    cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
            "trip_distance", "fare_amount", "passenger_count"]

    print("=== Rule 1: trip_distance <= 0 or null ===")
    raw_df.filter(
        F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0)
    ).select(cols).show(3, truncate=False)

    print("=== Rule 2: fare_amount < 0 ===")
    raw_df.filter(F.col("fare_amount") < 0).select(cols).show(3, truncate=False)

    print("=== Rule 5: dropoff_datetime <= pickup_datetime ===")
    raw_df.filter(
        F.col("tpep_pickup_datetime").isNotNull()  &
        F.col("tpep_dropoff_datetime").isNotNull() &
        (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
    ).select(cols).show(3, truncate=False)

    print("=== Custom Scenario: passenger_count null or 0 (imputed, not dropped) ===")
    raw_df.filter(
        F.col("passenger_count").isNull() | (F.col("passenger_count") == 0)
    ).select(cols).show(3, truncate=False)
else:
    print("No new files -- add files to data/inbox/ and rerun.")

## 9. Enrich with Zone Names

Join the zone lookup table twice (once for pickup, once for dropoff).

In [ ]:
if processed_dfs:
    new_data = reduce(lambda a, b: a.union(b), processed_dfs)

    pickup_zones = zone_lookup.select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough"),
    )
    dropoff_zones = zone_lookup.select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("Borough").alias("dropoff_borough"),
    )

    new_data = new_data.join(pickup_zones,  on="PULocationID",  how="left")
    new_data = new_data.join(dropoff_zones, on="DOLocationID", how="left")

    output_cols = [
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "PULocationID", "DOLocationID",
        "pickup_zone", "pickup_borough",
        "dropoff_zone", "dropoff_borough",
        "passenger_count", "trip_distance", "fare_amount",
        "trip_duration_minutes", "pickup_date",
        "source_file", "ingested_at",
    ]
    new_data = new_data.select(output_cols)
    print(f"New enriched rows ready: {new_data.count():,}")
else:
    print("No new data to enrich.")

## 10. Merge with Existing Output & Write

If `trips_enriched.parquet` already exists, union the new data with it and
dedup on the same key to guarantee no duplicates across runs.

In [ ]:
if processed_dfs:
    outbox_path = Path(OUTBOX_PATH)

    if outbox_path.exists():
        existing = spark.read.parquet(str(outbox_path))
        existing_count = existing.count()
        print(f"Existing output rows: {existing_count:,}")
        combined = existing.union(new_data)
    else:
        combined = new_data
        print("No existing output -- writing fresh.")

    # Dedup the combined dataset using the same key as per-file dedup
    combined = combined.withColumn("_fare_rounded", F.round(F.col("fare_amount"), 2))
    merge_key = [
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "PULocationID", "DOLocationID", "_fare_rounded",
    ]
    combined = combined.dropDuplicates(merge_key).drop("_fare_rounded")

    # Count before write so we don't trigger a second full computation
    final_count = combined.count()

    Path(OUTBOX_DIR).mkdir(parents=True, exist_ok=True)
    combined.write.mode("overwrite").parquet(OUTBOX_PATH)
    print(f"Output written : {OUTBOX_PATH}")
    print(f"Final row count: {final_count:,}")
else:
    final_count = 0
    print("No new data -- output unchanged.")

## 11. Update Manifest

Manifest is written **after** the output is confirmed written — if the write fails, the
files stay unregistered and will be reprocessed on the next run.

In [ ]:
if new_manifest_entries:
    manifest["processed_files"].extend(new_manifest_entries)
    save_manifest(MANIFEST_PATH, manifest)
    print(f"Manifest saved to {MANIFEST_PATH}")
    print(json.dumps(manifest, indent=2))
else:
    print("No new files processed -- manifest unchanged.")

## 12. Pipeline Summary

In [ ]:
SEP = "=" * 60
print(SEP)
print("PIPELINE SUMMARY")
print(SEP)

if new_manifest_entries:
    for entry in new_manifest_entries:
        dropped_clean = entry["row_count_raw"] - entry["row_count_after_clean"]
        dropped_dedup = entry["row_count_after_clean"] - entry["row_count_after_dedup"]
        print(f"\nFile: {entry['filename']}")
        print(f"  Raw rows:           {entry['row_count_raw']:>10,}")
        print(f"  After cleaning:     {entry['row_count_after_clean']:>10,}  (dropped {dropped_clean:,})")
        print(f"  After dedup:        {entry['row_count_after_dedup']:>10,}  (dropped {dropped_dedup:,})")
        print(f"  Passenger imputed:  {entry['imputed_passenger_count']:>10,}")
        print(f"  Global median used: {entry['global_median_used']:>10}")
    print(f"\nFinal output rows:    {final_count:>10,}")
else:
    print("No new files processed.")

print(f"\nOutput  : {OUTBOX_PATH}")
print(f"Manifest: {MANIFEST_PATH}")

_wall_time = time.time() - _pipeline_start
print(f"\nWall time: {_wall_time:.1f}s  ({_wall_time / 60:.2f} min)")
print(SEP)